In [1]:
import io
import pandas as pd
import requests
import os
import numpy as np
import matplotlib.pyplot as plt

url = "https://mydataaijournal.com/public/andy-data/olist-data/sla_breached.parquet"

print("Fetching dataset from mydataaijournal.com...")
response = requests.get(url)
response.raise_for_status()

df_sla = pd.read_parquet(io.BytesIO(response.content))
print(f"Successfully loaded {len(df_sla):,} rows into memory!")

Fetching dataset from mydataaijournal.com...
Successfully loaded 96,353 rows into memory!


In [2]:
df_sla.columns = df_sla.columns.str.lower()

In [3]:
df_sla.head()

,order_id,customer_id,order_purchase_at,order_estimated_delivery_at,order_delivered_customer_at,sla_delay_days,is_sla_breached,review_score
0,73fc7af87114b39712e6da79b0a377eb,41dcb106f807e993532d446263290104,2018-01-11 15:30:49,2018-02-02,2018-01-17 18:42:41,-16,0,4
1,a548910a1c6147796b98fdf73dbeba33,8a2e7ef9053dea531e4dc76bd6d853e6,2018-02-28 12:25:19,2018-03-14,2018-03-09 23:17:20,-5,0,5
2,f9e4b658b201a9f2ecdecbb34bed034b,e226dfed6544df5b7b87a48208690feb,2018-02-03 09:56:22,2018-03-09,2018-02-16 17:28:48,-21,0,5
3,658677c97b385a9be170737859d3511b,de6dff97e5f1ba84a3cd9a3bc97df5f6,2017-04-09 17:41:13,2017-05-10,2017-04-20 09:08:35,-20,0,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,5986b333ca0d44534a156a52a8e33a83,2018-02-10 10:59:03,2018-03-09,2018-02-28 16:33:35,-9,0,5


In [4]:
x_real = df_sla["sla_delay_days"]
y_real = df_sla["review_score"]



In [5]:
import numpy as np
import pandas as pd
import scipy.stats as stats


def parametric_regression(x, y, alpha=0.05):
    # 1. Fit linear regression model
    res = stats.linregress(x, y)
    obs_slope = res.slope
    p_value = res.pvalue  # Two-tailed p-value based on Student's t-distribution
    std_err = res.stderr

    # 2. Compute 95% Confidence Interval for the slope
    n = len(x)
    df = n - 2  # Degrees of freedom for simple linear regression
    t_crit = stats.t.ppf(1 - alpha / 2, df=df)  # Two-tailed critical value

    margin_of_error = t_crit * std_err
    ci_lower = obs_slope - margin_of_error
    ci_upper = obs_slope + margin_of_error

    return p_value, obs_slope, std_err, (ci_lower, ci_upper)


# Run parametric regression
p_value, obs_slope, std_err, (ci_lower, ci_upper) = parametric_regression(
    x_real.values, y_real.values
)

print("--- Parametric Linear Regression Results ---")
print(f"Observed Slope (β1): {obs_slope:.5f}")
print(f"Standard Error:      {std_err:.5f}")
print(f"Parametric p-value:  {p_value:.5e}")
print(f"95% CI for Slope:    [{ci_lower:.5f}, {ci_upper:.5f}]")

--- Parametric Linear Regression Results ---
Observed Slope (β1): -0.03393
Standard Error:      0.00039
Parametric p-value:  0.00000e+00
95% CI for Slope:    [-0.03470, -0.03316]
